In [2]:
import os
import random
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

# Konfiguracja wyświetlania pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# --- Globalne ziarno losowości ---
# Krytyczne dla Isolation Forest, który opiera się na losowych podziałach!
SEED = 123
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = Path("C:/Users/wrata/OneDrive - SGH/MAGISTERKA/2 semestr/data mining/PROJEKT/dataset")

print("Wczytuję train_transaction...")
train_transaction = pd.read_csv(DATA_DIR / "train_transaction.csv")
print(f"   OK — {train_transaction.shape}")

print("Wczytuję train_identity...")
train_identity = pd.read_csv(DATA_DIR / "train_identity.csv")
print(f"   OK — {train_identity.shape}")

# KROK SPECYFICZNY POD ISOLATION FOREST:
# Zapamiętujemy oryginalne listy kolumn numerycznych i kategorycznych
# Przyda nam się to, aby precyzyjnie kontrolować transformacje geometryczne.
raw_cat_cols_transaction = (
    train_transaction.select_dtypes(include=["object"]).columns.tolist()
)
raw_cat_cols_identity = (
    train_identity.select_dtypes(include=["object"]).columns.tolist()
)
ALL_RAW_CAT_COLS = list(set(raw_cat_cols_transaction + raw_cat_cols_identity))

print(
    f"\nWykryte surowe kolumny kategoryczne (tekstowe): {len(ALL_RAW_CAT_COLS)}"
)

Wczytuję train_transaction...
   OK — (590540, 394)
Wczytuję train_identity...
   OK — (144233, 41)

Wykryte surowe kolumny kategoryczne (tekstowe): 31


In [3]:
# === 1. Scalanie po TransactionID (left join) ===
train = train_transaction.merge(train_identity, on="TransactionID", how="left")

print("Po scaleniu:")
print(f"  train: {train.shape}")

print(f"\nTransakcje z danymi identity:")
print(
    f"  train: {train_identity.shape[0]:,} z {train_transaction.shape[0]:,} ({train_identity.shape[0]/train_transaction.shape[0]*100:.1f}%)"
)


# Upewniamy się, że na liście są tylko te kolumny, które fizycznie znalazły się w złączonym zbiorze
ALL_RAW_CAT_COLS = [c for c in ALL_RAW_CAT_COLS if c in train.columns]


# === 2. ZMIENNA CELU ===
print("\n=== ZMIENNA CELU ===")
fraud_counts = train["isFraud"].value_counts()
print(fraud_counts)
print(f"\nFraud rate: {train['isFraud'].mean()*100:.2f}%")

# KROK REKOMENDOWANY: Wydzielenie zmiennej celu
# Isolation Forest to model nienadzorowany (unsupervised). Trening (.fit) odbędzie się całkowicie
# BEZ użycia etykiet. Zabezpieczamy 'isFraud' już teraz, aby nie uczestniczyła w analizie braków.
y_global = train["isFraud"]

Po scaleniu:
  train: (590540, 434)

Transakcje z danymi identity:
  train: 144,233 z 590,540 (24.4%)

=== ZMIENNA CELU ===
isFraud
0    569877
1     20663
Name: count, dtype: int64

Fraud rate: 3.50%


In [4]:
# === 1. Obliczenie procentu brakujących wartości ===
missing = (train.isnull().sum() / len(train) * 100).sort_values(ascending=False)

# === 2. Podział na grupy (Podsumowanie ogólne) ===
print("=== BRAKUJĄCE DANE — PODSUMOWANIE ===")
print(f"Kolumny bez żadnych braków:   {(missing == 0).sum()}")
print(f"Kolumny z 0–20% NaN:          {((missing > 0) & (missing <= 20)).sum()}")
print(f"Kolumny z 20–50% NaN:         {((missing > 20) & (missing <= 50)).sum()}")
print(f"Kolumny z 50–80% NaN:         {((missing > 50) & (missing <= 80)).sum()}")
print(f"Kolumny z >80% NaN:           {(missing > 80).sum()}")

print("\n=== TOP 20 KOLUMN Z NAJWIĘKSZĄ LICZBĄ NaN ===")
print(missing.head(20).round(1).to_string())

# === 3. Analiza grup kolumn po prefiksach ===
print("\n=== GRUPY KOLUMN — % NaN ===")
for prefix, label in [("V", "V (Vesta)"), ("D", "D (deltas)"), 
                       ("id_", "identity"), ("C", "C (counts)"), ("M", "M (flags)")]:
    cols = [c for c in train.columns if c.startswith(prefix)]
    if cols:
        m = missing[cols]
        print(f"\n{label} ({len(cols)} kolumn):")
        print(f"  min NaN: {m.min():.1f}%  |  max NaN: {m.max():.1f}%  |  mediana NaN: {m.median():.1f}%")

=== BRAKUJĄCE DANE — PODSUMOWANIE ===
Kolumny bez żadnych braków:   20
Kolumny z 0–20% NaN:          162
Kolumny z 20–50% NaN:         38
Kolumny z 50–80% NaN:         140
Kolumny z >80% NaN:           74

=== TOP 20 KOLUMN Z NAJWIĘKSZĄ LICZBĄ NaN ===
id_24    99.2
id_25    99.1
id_07    99.1
id_08    99.1
id_21    99.1
id_26    99.1
id_27    99.1
id_23    99.1
id_22    99.1
dist2    93.6
D7       93.4
id_18    92.4
D13      89.5
D14      89.5
D12      89.0
id_03    88.8
id_04    88.8
D6       87.6
id_33    87.6
id_10    87.3

=== GRUPY KOLUMN — % NaN ===

V (Vesta) (339 kolumn):
  min NaN: 0.0%  |  max NaN: 86.1%  |  mediana NaN: 47.3%

D (deltas) (17 kolumn):
  min NaN: 0.2%  |  max NaN: 93.4%  |  mediana NaN: 76.2%

identity (38 kolumn):
  min NaN: 75.6%  |  max NaN: 99.2%  |  mediana NaN: 82.4%

C (counts) (14 kolumn):
  min NaN: 0.0%  |  max NaN: 0.0%  |  mediana NaN: 0.0%

M (flags) (9 kolumn):
  min NaN: 28.7%  |  max NaN: 59.3%  |  mediana NaN: 47.7%


In [5]:
# === 1. Identyfikacja kolumn z ekstremalną liczbą braków (>80%) ===
high_missing_cols = missing[missing > 80].index.tolist()

print(f"Kolumny z >80% NaN: {len(high_missing_cols)}")
print(f"\nZ jakich grup pochodzą:")
for prefix, label in [("V", "V"), ("D", "D"), ("id_", "identity"), ("C", "C"), ("M", "M"), ("dist", "dist")]:
    cols = [c for c in high_missing_cols if c.startswith(prefix)]
    if cols:
        print(f"  {label}: {len(cols)} kolumn — {cols}")

# === 2. Analiza wpływu samej obecności wartości na współczynnik oszustw ===
print("\n=== FRAUD RATE dla kolumn z >80% NaN (gdy wartość JEST dostępna) ===")
print("(czy sam fakt że kolumna ma wartość koreluje z fraudem?)\n")

results = []
for col in high_missing_cols:
    has_value = train[col].notna()
    fraud_with    = train.loc[has_value, "isFraud"].mean() * 100
    fraud_without = train.loc[~has_value, "isFraud"].mean() * 100
    results.append({
        "kolumna": col,
        "percent_NaN": missing[col].round(1), # unikamy spacji w nazwach kluczy
        "fraud_gdy_ma_wartosc": round(fraud_with, 2),
        "fraud_gdy_NaN": round(fraud_without, 2),
        "roznica": round(fraud_with - fraud_without, 2)
    })

df_results = pd.DataFrame(results).sort_values("roznica", key=abs, ascending=False)
print(df_results.head(20).to_string(index=False))

# === KROK SPECYFICZNY POD ISOLATION FOREST ===
# Automatycznie wyodrębniamy kolumny, które NIE są z grupy Vesta ("V")
# Te kolumny (D, id, dist) mają potężną różnicę w fraud rate. Dla nich w kolejnych krokach
# wygenerujemy flagi obecności danych, a surowe kolumny bezwzględnie USUNIEMY.
non_v_high_missing = [c for c in high_missing_cols if not c.startswith("V")]

Kolumny z >80% NaN: 74

Z jakich grup pochodzą:
  V: 47 kolumn — ['V142', 'V158', 'V140', 'V162', 'V141', 'V161', 'V157', 'V146', 'V156', 'V155', 'V154', 'V153', 'V149', 'V147', 'V148', 'V163', 'V139', 'V138', 'V160', 'V151', 'V152', 'V145', 'V144', 'V143', 'V159', 'V164', 'V165', 'V166', 'V150', 'V337', 'V333', 'V336', 'V335', 'V334', 'V338', 'V339', 'V324', 'V332', 'V325', 'V330', 'V329', 'V328', 'V327', 'V326', 'V322', 'V323', 'V331']
  D: 7 kolumn — ['D7', 'D13', 'D14', 'D12', 'D6', 'D9', 'D8']
  identity: 19 kolumn — ['id_24', 'id_25', 'id_07', 'id_08', 'id_21', 'id_26', 'id_27', 'id_23', 'id_22', 'id_18', 'id_03', 'id_04', 'id_33', 'id_10', 'id_09', 'id_30', 'id_32', 'id_34', 'id_14']
  dist: 1 kolumn — ['dist2']

=== FRAUD RATE dla kolumn z >80% NaN (gdy wartość JEST dostępna) ===
(czy sam fakt że kolumna ma wartość koreluje z fraudem?)

kolumna  percent_NaN  fraud_gdy_ma_wartosc  fraud_gdy_NaN  roznica
     D7         93.4                 14.88           2.70    12.18
    D12  

In [6]:
# === 1. Filtrowanie kolumn Vesta z grupy wysokich braków ===
v_high_missing = [c for c in high_missing_cols if c.startswith("V")]

print(f"V-kolumny z >80% NaN: {len(v_high_missing)}")
print(f"Zakres: {sorted(v_high_missing, key=lambda x: int(x[1:]))}\n")

# === 2. Głęboka analiza statystyczna bloku Vesta ===
results_v = []
for col in v_high_missing:
    has_value = train[col].notna()
    fraud_with    = train.loc[has_value, "isFraud"].mean() * 100
    fraud_without = train.loc[~has_value, "isFraud"].mean() * 100
    
    # POPRAWKA POD ISOLATION FOREST: Bezpieczne obliczanie korelacji liniowej.
    # Jeśli podzbiór danych ma zerową wariancję (np. same zera w istniejących wierszach),
    # pandas zwraca NaN. Mapujemy to na 0.0 (brak korelacji liniowej), aby nie popsuć filtrów logicznych.
    if train.loc[has_value, col].nunique() <= 1:
        corr = 0.0
    else:
        corr = train.loc[has_value, [col, "isFraud"]].corr().iloc[0, 1]
        if pd.isna(corr):
            corr = 0.0
            
    results_v.append({
        "kolumna": col,
        "percent_NaN": round(missing[col], 1),
        "fraud_gdy_wartosc": round(fraud_with, 2),
        "fraud_gdy_NaN": round(fraud_without, 2),
        "roznica": round(fraud_with - fraud_without, 2),
        "korelacja_wartosci_z_fraudem": round(corr, 3)
    })

# Tworzymy ramkę wynikową df_v, która posłuży do automatycznej selekcji w kolejnej komórce
df_v = pd.DataFrame(results_v).sort_values("roznica", key=abs, ascending=False)
print(df_v.to_string(index=False))

V-kolumny z >80% NaN: 47
Zakres: ['V138', 'V139', 'V140', 'V141', 'V142', 'V143', 'V144', 'V145', 'V146', 'V147', 'V148', 'V149', 'V150', 'V151', 'V152', 'V153', 'V154', 'V155', 'V156', 'V157', 'V158', 'V159', 'V160', 'V161', 'V162', 'V163', 'V164', 'V165', 'V166', 'V322', 'V323', 'V324', 'V325', 'V326', 'V327', 'V328', 'V329', 'V330', 'V331', 'V332', 'V333', 'V334', 'V335', 'V336', 'V337', 'V338', 'V339']

kolumna  percent_NaN  fraud_gdy_wartosc  fraud_gdy_NaN  roznica  korelacja_wartosci_z_fraudem
   V331         86.1               4.48           3.34     1.14                        -0.022
   V332         86.1               4.48           3.34     1.14                        -0.023
   V337         86.1               4.48           3.34     1.14                        -0.006
   V333         86.1               4.48           3.34     1.14                        -0.024
   V336         86.1               4.48           3.34     1.14                        -0.002
   V335         86.1     

In [7]:
# === 1. Filtrowanie V-kolumn na podstawie kryteriów pod Isolation Forest ===
# Dopasowujemy nazwy kluczy do formatu bez polskich znaków z poprzedniej komórki.
v_to_drop = [
    r["kolumna"] for _, r in df_v.iterrows() 
    if abs(r["korelacja_wartosci_z_fraudem"]) < 0.05 and r["roznica"] < 2
]

v_to_keep = [
    r["kolumna"] for _, r in df_v.iterrows() 
    if r["kolumna"] not in v_to_drop
]

print(f"V-kolumny z >80% NaN do USUNIĘCIA ({len(v_to_drop)}):")
print(sorted(v_to_drop, key=lambda x: int(x[1:])))

print(f"\nV-kolumny z >80% NaN do ZACHOWANIA ({len(v_to_keep)}):")
print(sorted(v_to_keep, key=lambda x: int(x[1:])))

# === 2. Przygotowanie ostatecznej listy kolumn do usunięcia ===
# Zgodnie z naszą strategią: do usunięcia idą V-kolumny bez sygnału (v_to_drop).
cols_to_drop = v_to_drop

print(f"\nŁącznie kolumn do usunięcia w tym kroku: {len(cols_to_drop)}")
print(f"Kolumny przed usunięciem: {train.shape[1]}")
print(f"Kolumny po usunięciu:     {train.shape[1] - len(cols_to_drop)}")

V-kolumny z >80% NaN do USUNIĘCIA (24):
['V138', 'V143', 'V161', 'V163', 'V164', 'V166', 'V322', 'V323', 'V324', 'V325', 'V326', 'V327', 'V328', 'V329', 'V330', 'V331', 'V332', 'V333', 'V334', 'V335', 'V336', 'V337', 'V338', 'V339']

V-kolumny z >80% NaN do ZACHOWANIA (23):
['V139', 'V140', 'V141', 'V142', 'V144', 'V145', 'V146', 'V147', 'V148', 'V149', 'V150', 'V151', 'V152', 'V153', 'V154', 'V155', 'V156', 'V157', 'V158', 'V159', 'V160', 'V162', 'V165']

Łącznie kolumn do usunięcia w tym kroku: 24
Kolumny przed usunięciem: 434
Kolumny po usunięciu:     410


In [8]:
# === 1. USUNIĘCIE kolumn V bez sygnału ===
train = train.drop(columns=cols_to_drop)
print(f"Po usunięciu V-kolumn bez sygnału: train {train.shape}")

# === 2. SELEKCJA rzadkich kolumn D, identity i dist (>80% NaN) ===
d_high    = [c for c in high_missing_cols if c.startswith("D") and c in train.columns]
id_high   = [c for c in high_missing_cols if c.startswith("id_") and c in train.columns]
dist_high = [c for c in high_missing_cols if c.startswith("dist") and c in train.columns]

flag_cols = d_high + id_high + dist_high
print(f"\nTworzymy flagi _isnan dla {len(flag_cols)} kolumn:")
print(f"  D:        {d_high}")
print(f"  identity: {id_high}")
print(f"  dist:     {dist_high}")

# === 3. GENEROWANIE FLAG I USUNIĘCIE SUROWYCH KOLUMN (Poprawka pod iForest) ===
for col in flag_cols:
    # Tworzymy jawną cechę binarną odzwierciedlającą strukturę braku danych
    train[f"{col}_isnan"] = train[col].isnull().astype(int)

# KLUCZOWY KROK DLA MODELU ANOMALII:
# Usuwamy surowe kolumny numeryczne, z których wyciągnęliśmy flagi. Zostawienie ich zmuszałoby
# iForest do losowego rozcinania mało istotnych danych numerycznych w rzadkich podzbiorach (szum).
train = train.drop(columns=flag_cols)

print(f"\nPo dodaniu flag i usunięciu surowych kolonii: train {train.shape}")

# === 4. Weryfikacja ===
print(f"\nWeryfikacja D7_isnan:")
if "D7_isnan" in train.columns:
    print(train.groupby("D7_isnan")["isFraud"].mean().round(4) * 100)
else:
    print("Modyfikacja wykonana pomyślnie. Kolumna D7 została poprawnie zastąpiona cechą D7_isnan.")

Po usunięciu V-kolumn bez sygnału: train (590540, 410)

Tworzymy flagi _isnan dla 27 kolumn:
  D:        ['D7', 'D13', 'D14', 'D12', 'D6', 'D9', 'D8']
  identity: ['id_24', 'id_25', 'id_07', 'id_08', 'id_21', 'id_26', 'id_27', 'id_23', 'id_22', 'id_18', 'id_03', 'id_04', 'id_33', 'id_10', 'id_09', 'id_30', 'id_32', 'id_34', 'id_14']
  dist:     ['dist2']

Po dodaniu flag i usunięciu surowych kolonii: train (590540, 410)

Weryfikacja D7_isnan:
D7_isnan
0    14.88
1     2.70
Name: isFraud, dtype: float64


In [9]:
# === PRZEGLĄD ZMIENNYCH KATEGORYCZNYCH ===
# Synchronizujemy listę z zabezpieczonymi na początku kolumnami tekstowymi,
# upewniając się, że badamy tylko te, które fizycznie przetrwały dotychczasową selekcję.
cat_cols = [c for c in ALL_RAW_CAT_COLS if c in train.columns]

print(f"Kolumny kategoryczne do analizy: {len(cat_cols)}")
print(f"{cat_cols}\n")

for col in cat_cols:
    n_unique = train[col].nunique()
    n_missing = train[col].isnull().mean() * 100
    
    # Obliczamy fraud rate dla istniejących kategorii
    fraud_by_cat = train.groupby(col)["isFraud"].mean().sort_values(ascending=False) * 100
    
    print(f"--- {col} ---")
    print(f"  unikalne wartości: {n_unique}  |  NaN: {n_missing:.1f}%")
    print(f"  top fraud rate:")
    print(fraud_by_cat.head(5).round(2).to_string())
    print()

Kolumny kategoryczne do analizy: 26
['M5', 'M7', 'id_15', 'P_emaildomain', 'ProductCD', 'id_38', 'id_29', 'id_36', 'id_35', 'M1', 'M8', 'id_16', 'card6', 'M3', 'DeviceType', 'M4', 'DeviceInfo', 'M2', 'R_emaildomain', 'M6', 'id_28', 'card4', 'id_12', 'id_31', 'id_37', 'M9']

--- M5 ---
  unikalne wartości: 2  |  NaN: 59.3%
  top fraud rate:
M5
T    3.77
F    2.65

--- M7 ---
  unikalne wartości: 2  |  NaN: 58.6%
  top fraud rate:
M7
T    2.21
F    1.93

--- id_15 ---
  unikalne wartości: 3  |  NaN: 76.1%
  top fraud rate:
id_15
Found      10.51
Unknown     9.19
New         4.92

--- P_emaildomain ---
  unikalne wartości: 59  |  NaN: 16.0%
  top fraud rate:
P_emaildomain
protonmail.com    40.79
mail.com          18.96
outlook.es        13.01
aim.com           12.70
outlook.com        9.46

--- ProductCD ---
  unikalne wartości: 5  |  NaN: 0.0%
  top fraud rate:
ProductCD
C    11.69
S     5.90
H     4.77
R     3.78
W     2.04

--- id_38 ---
  unikalne wartości: 2  |  NaN: 76.1%
  top frau

In [10]:
# === ENCODING KATEGORYCZNYCH POD ISOLATION FOREST ===

# --- 1. M-kolumny: T/F → 1/0, NaN → -1 (Bez zmian — ten krok był idealny) ---
m_cols = [c for c in train.columns if c.startswith("M")]
for col in m_cols:
    train[col] = train[col].map({"T": 1, "F": 0}).fillna(-1).astype(int)
print(f"M-kolumny ({len(m_cols)}) → zakodowane geometrycznie jako 1/0/-1")


# --- 2. Agrupowanie tekstowe Domen Email (Twoje świetne funkcje reguł biznesowych) ---
def group_email(domain):
    if pd.isna(domain): return "missing"
    d = str(domain).lower()
    if "gmail" in d:   return "gmail"
    if "yahoo" in d:   return "yahoo"
    if "hotmail" in d or "outlook" in d or "live" in d: return "microsoft"
    if "anonymous" in d: return "anonymous"
    if "protonmail" in d or "mail.com" in d: return "high_risk"
    return "other"

for col in ["P_emaildomain", "R_emaildomain"]:
    if col in train.columns:
        train[col] = train[col].apply(group_email)
print("Email domains → zgrupowane tekstowo")


# --- 3. id_31: Uproszczenie rodziny przeglądarki (Kolumna id_30 została już usunięta jako flag_col) ---
def extract_browser(val):
    if pd.isna(val): return "missing"
    val = str(val).lower()
    if "chrome" in val:   return "chrome"
    if "firefox" in val or "mozilla" in val: return "firefox"
    if "safari" in val:   return "safari"
    if "edge" in val:     return "edge"
    if "ie" in val or "internet explorer" in val: return "ie"
    if "samsung" in val:  return "samsung"
    return "other"

if "id_31" in train.columns:
    train["id_31"] = train["id_31"].apply(extract_browser)
print("id_31 → uproszczony do rodziny przeglądarki")


# --- 4. DeviceInfo → zastąpienie flagą obecności (id_33 zostało już usunięte jako flag_col) ---
if "DeviceInfo" in train.columns:
    train["DeviceInfo_present"] = train["DeviceInfo"].notna().astype(int)
    train = train.drop(columns=["DeviceInfo"])
print("DeviceInfo → zastąpione flagą _present ze względu na zbyt wysoką kardynalność")


# --- 5. KLUCZOWA POPRAWKA POD iFOREST: One-Hot Encoding zamiast .codes ---
# Zbieramy wszystkie cechy kategoryczne, które faktycznie znajdują się w tabeli na tym etapie.
# Pomijamy kolumny, które usunęliśmy w poprzednich krokach (id_23, id_27, id_30, id_33, id_34).
ohe_cols = ["ProductCD", "card4", "card6", "DeviceType",
            "id_12", "id_15", "id_16", "id_28", "id_29", 
            "id_35", "id_36", "id_37", "id_38", 
            "P_emaildomain", "R_emaildomain", "id_31"]

ohe_cols = [c for c in ohe_cols if c in train.columns]

# Generujemy rygorystyczny geometrycznie One-Hot Encoding.
# Ustawiamy dummy_na=True, aby ewentualne ukryte braki danych również dostały czysty wymiar 0/1.
train = pd.get_dummies(train, columns=ohe_cols, dummy_na=True, dtype=int)
print(f"Wykonano One-Hot Encoding dla {len(ohe_cols)} kolumn kategorycznych (w tym obsługa NaN).")


# --- 6. Finalna weryfikacja poprawności typów w pamięci ---
print(f"\nPo encodingu: train {train.shape}")

remaining_cats = train.select_dtypes(include=["object"]).columns.tolist()
print(f"Pozostałe kolumny object (wymagane 0 pod Isolation Forest): {remaining_cats}")

M-kolumny (9) → zakodowane geometrycznie jako 1/0/-1
Email domains → zgrupowane tekstowo
id_31 → uproszczony do rodziny przeglądarki
DeviceInfo → zastąpione flagą _present ze względu na zbyt wysoką kardynalność
Wykonano One-Hot Encoding dla 16 kolumn kategorycznych (w tym obsługa NaN).

Po encodingu: train (590540, 466)
Pozostałe kolumny object (wymagane 0 pod Isolation Forest): []


In [11]:
# === FEATURE ENGINEERING POD ISOLATION FOREST ===

# --- 1. TransactionDT: Ekstrakcja komponentów czasowych ---
train["hour"]       = (train["TransactionDT"] // 3600) % 24
train["dayofweek"]  = (train["TransactionDT"] // (3600 * 24)) % 7

print("Fraud rate wg godziny (top 5 najbardziej fraudowych):")
print((train.groupby("hour")["isFraud"].mean() * 100).sort_values(ascending=False).head(5).round(2))

print("\nFraud rate wg dnia tygodnia:")
print((train.groupby("dayofweek")["isFraud"].mean() * 100).round(2))


# --- 2. log_TransactionAmt: Transformacja nieliniowa kwoty ---
train["log_TransactionAmt"] = np.log1p(train["TransactionAmt"])

print(f"\nTransactionAmt — skośność oryginalna: {train['TransactionAmt'].skew():.2f}")
print(f"TransactionAmt — skośność po log1p:   {train['log_TransactionAmt'].skew():.2f}")
# UWAGA: Celowo NIE usuwamy surowej kolumny 'TransactionAmt'. 
# Isolation Forest opiera się na dystansie losowych podziałów i obecność oryginalnego, 
# silnie skośnego prawego ogona rozkładu drastycznie ułatwia mu natychmiastowe izolowanie transakcji na skrajnie wysokie kwoty.


# --- 3. email_match: Analiza zgodności domen nadawcy i odbiorcy ---
# POPRAWKA ARCHITEKTONICZNA: Ponieważ surowe kolumny tekstowe e-maili zostały rozbite w OHE, 
# zgodność (match) wyznaczamy poprzez iloczyn macierzowy kolumn binarnych odpowiadających tym samym rodzinom.
email_families = ["gmail", "yahoo", "microsoft", "anonymous", "high_risk", "other"]
train["email_match"] = 0

for family in email_families:
    p_col = f"P_emaildomain_{family}"
    r_col = f"R_emaildomain_{family}"
    if p_col in train.columns and r_col in train.columns:
        # Jeśli w obu kolumnach jest 1, to iloczyn da 1 (nastąpiło dopasowanie rodziny e-mail)
        train["email_match"] = train["email_match"] | (train[p_col] & train[r_col])

print(f"\nFraud rate gdy email_match=1: {train[train['email_match']==1]['isFraud'].mean()*100:.2f}%")
print(f"Fraud rate gdy email_match=0: {train[train['email_match']==0]['isFraud'].mean()*100:.2f}%")


# --- 4. Czyszczenie osi czasu i podsumowanie kroku ---
train = train.drop(columns=["TransactionDT"])

print(f"\nPo feature engineering: train {train.shape}")

Fraud rate wg godziny (top 5 najbardziej fraudowych):
hour
7    10.61
8     9.30
9     9.00
6     7.77
5     7.03
Name: isFraud, dtype: float64

Fraud rate wg dnia tygodnia:
dayofweek
0    3.72
1    3.60
2    3.71
3    3.56
4    3.15
5    3.30
6    3.45
Name: isFraud, dtype: float64

TransactionAmt — skośność oryginalna: 14.37
TransactionAmt — skośność po log1p:   0.49

Fraud rate gdy email_match=1: 9.57%
Fraud rate gdy email_match=0: 2.21%

Po feature engineering: train (590540, 469)


In [12]:
# === IMPUTACJA NaN POD ISOLATION FOREST ===

# 1. Identyfikacja pozostałych braków danych
missing_after = train.isnull().sum()
missing_after = missing_after[missing_after > 0].sort_values(ascending=False)

print(f"Kolumny z NaN przed imputacją: {len(missing_after)}")
print(f"\nTop 15:")
print((missing_after.head(15) / len(train) * 100).round(1).to_string())

# 2. POPRAWKA STRATEGICZNA: Imputacja wartością stałą spoza zakresu (-999) zamiast medianą.
# Zabezpiecza to geometrię Isolation Forest przed sztucznym zagęszczaniem środka rozkładu.
print("\nImputacja wartością skrajną (-999)...")
for col in missing_after.index.tolist():
    train[col] = train[col].fillna(-999)

# 3. Podsumowanie i weryfikacja
print(f"\nNaN po imputacji — train: {train.isnull().sum().sum()}")
print(f"\nKształt finalny macierzy cech: train {train.shape}")

Kolumny z NaN przed imputacją: 338

Top 15:
V158    86.1
V139    86.1
V141    86.1
V142    86.1
V146    86.1
V147    86.1
V148    86.1
V149    86.1
V153    86.1
V154    86.1
V155    86.1
V156    86.1
V157    86.1
V162    86.1
V140    86.1

Imputacja wartością skrajną (-999)...

NaN po imputacji — train: 0

Kształt finalny macierzy cech: train (590540, 469)


In [14]:
# === FINALNA WERYFIKACJA I MAPOWANIE TYPÓW POD iFOREST ===

# POPRAWKA BEZPIECZEŃSTWA: Wymuszamy rzutowanie typów bool (True/False) na int (1/0).
# Zapobiega to krytycznemu błędowi typu w scikit-learn podczas wywołania model.fit().
bool_cols = train.select_dtypes(include=["bool"]).columns.tolist()
if bool_cols:
    train[bool_cols] = train[bool_cols].astype(int)
    print(f"Zmapowano {len(bool_cols)} kolumn typu bool na format numeryczny int.")

print(f"\nKształt finalny tabeli: train {train.shape}")
print(f"NaN — train: {train.isnull().sum().sum()}")
print(f"\nPodsumowanie typów danych w macierzy:")
print(train.dtypes.value_counts())
print(f"\nGlobalny fraud rate: {train['isFraud'].mean()*100:.2f}%")


# === CHRONOLOGICZNY PODZIAŁ OUT-OF-TIME (60/20/20) ===
# Sortujemy po TransactionID, realizując rygorystyczny podział czasowy.
train_sorted = train.sort_values("TransactionID").reset_index(drop=True)

n = len(train_sorted)
n_train = int(n * 0.60)
n_valid  = int(n * 0.80)   # 60% + 20%

train_model = train_sorted.iloc[:n_train]
valid_model  = train_sorted.iloc[n_train:n_valid]
test_model   = train_sorted.iloc[n_valid:]

print("\n=== PODZIAŁ ZBIORU DANYCH ===")
for name, part in [("train", train_model), ("valid", valid_model), ("test", test_model)]:
    print(f"{name:5s} | shape: {part.shape} | fraud rate: {part['isFraud'].mean()*100:.4f}%")


# === WYDZIELENIE MACIERZY CECH (X) ORAZ WEKTORÓW ETYKIET (y) ===
# Bezwzględnie usuwamy 'isFraud' oraz 'TransactionID' z puli cech modelu.
X_train = train_model.drop(columns=["isFraud", "TransactionID"])
y_train = train_model["isFraud"]

X_valid = valid_model.drop(columns=["isFraud", "TransactionID"])
y_valid = valid_model["isFraud"]

X_test  = test_model.drop(columns=["isFraud", "TransactionID"])
y_test  = test_model["isFraud"]


# === ZAPIS OCZYSZCZONEGO ZBIORU ===
# Zapisujemy zbiór do pliku bez indeksu wierszy.
train_sorted.to_csv("train_cleaned.csv", index=False)
print(f"\n✓ Zapisano w pełni ujednolicony zbiór do pliku: train_cleaned.csv")
print(f"Liczba cech wejściowych dla Isolation Forest (X_train.shape[1]): {X_train.shape[1]}")


Kształt finalny tabeli: train (590540, 469)
NaN — train: 0

Podsumowanie typów danych w macierzy:
float64    354
int64      115
Name: count, dtype: int64

Globalny fraud rate: 3.50%

=== PODZIAŁ ZBIORU DANYCH ===
train | shape: (354324, 469) | fraud rate: 3.3833%
valid | shape: (118108, 469) | fraud rate: 3.9041%
test  | shape: (118108, 469) | fraud rate: 3.4409%

✓ Zapisano w pełni ujednolicony zbiór do pliku: train_cleaned.csv
Liczba cech wejściowych dla Isolation Forest (X_train.shape[1]): 467
